<div dir="ltr" style="direction:ltr; text-align:left;">

# [2] Analysis Objective

### 2.1 Problem Statement
Chronic diabetes progression leads to severe microvascular and cardiovascular complications if high-risk patients are not identified early. However, routine clinical blood panels contain strongly correlated lipid measurements (`Total_Cholesterol`, `LDL_Cholesterol`, `HDL_Cholesterol`, and `Total_HDL_Ratio`), which cause severe **multicollinearity** and unstable parameter estimates in standard unregularized regression models.

### 2.2 Main Objectives of the Regression Analysis
1. **Predictive Objective (Accurate Risk Forecasting):**
   - Build, tune, and compare multiple **Linear Regression variations** — **Simple Linear Regression**, **Multiple Ordinary Least Squares (OLS)**, **Ridge ($L_2$) Regression**, **Lasso ($L_1$) Regression**, **ElasticNet ($L_1 + L_2$) Regression**, and **Polynomial Degree-2 Ridge Regression** — to accurately predict a patient's 1-year `Disease_Progression` score from baseline clinical biomarkers.
2. **Explanatory Objective (Biomarker Attribution & Collinearity Resolution):**
   - Quantify the standardized effect size ($\beta$) of each modifiable physiological factor (`BMI`, `Log_Triglycerides`, `Blood_Pressure`, and `HDL_Cholesterol`) while using $L_1$ and $L_2$ regularization to eliminate redundant collinear predictors.
3. **Decision-Support Objective for Stakeholders:**
   - Deliver a transparent, clinically interpretable risk-scoring model that helps healthcare providers prioritize weight management, triglyceride control, and antihypertensive therapy.

</div>

In [1]:
# =============================================================================
# [2] ANALYSIS OBJECTIVE — PREPROCESSING & EXPERIMENTAL DESIGN (LTR + CONTRAST)
# =============================================================================
import math, random, statistics
from IPython.display import display, HTML

random.seed(42)
FEATURES = [
    "Age", "Sex", "BMI", "Blood_Pressure", "Total_Cholesterol",
    "LDL_Cholesterol", "HDL_Cholesterol", "Total_HDL_Ratio", "Log_Triglycerides", "Blood_Sugar"
]
DATA_ROWS = []
for _ in range(442):
    age = round(random.gauss(48.5, 13.1), 1)
    sex = 1.0 if random.random() < 0.53 else 2.0
    bmi = round(max(18.0, random.gauss(26.4, 4.4)), 2)
    bp = round(max(62.0, random.gauss(94.6, 13.8)), 2)
    hdl = round(max(22.0, random.gauss(49.8, 12.9 - 0.15 * (bmi - 26.4))), 2)
    ldl = round(max(55.0, random.gauss(115.4 + 1.2 * (bmi - 26.4), 30.4)), 2)
    ltg = round(max(3.2, random.gauss(4.64 + 0.04 * (bmi - 26.4), 0.52)), 2)
    tc = round(hdl + ldl + math.exp(ltg) * 0.20, 2)
    tch = round(tc / hdl, 2)
    glu = round(max(60.0, random.gauss(91.3 + 0.8 * (bmi - 26.4), 11.5)), 2)
    prog = (
        152.1 + 5.8 * (bmi - 26.4) + 1.15 * (bp - 94.6) + 42.5 * (ltg - 4.64)
        - 0.95 * (hdl - 49.8) + 0.35 * (glu - 91.3) - 18.0 * (sex - 1.47)
        + 0.90 * (bmi - 26.4) * (ltg - 4.64) + random.gauss(0, 48.0)
    )
    DATA_ROWS.append([age, sex, bmi, bp, tc, ldl, hdl, tch, ltg, glu, round(min(346.0, max(25.0, prog)), 1)])

train_rows, test_rows = DATA_ROWS[:353], DATA_ROWS[353:]
f_means = [statistics.mean([r[j] for r in train_rows]) for j in range(10)]
f_stds = [statistics.stdev([r[j] for r in train_rows]) for j in range(10)]

html_obj = f"""
<div dir='ltr' style='direction:ltr; text-align:left; background-color:#ffffff; color:#111111; padding:16px; border:1px solid #cccccc; border-radius:8px; font-family:Arial, sans-serif;'>
  <h3 style='color:#0f2942; margin-top:0;'>Table 2.1: Regression Analysis Workflow & Preprocessing Design</h3>
  <table dir='ltr' style='direction:ltr; text-align:left; border-collapse:collapse; width:100%; font-size:13px; color:#111111; background-color:#ffffff;'>
    <thead>
      <tr style='background-color:#0f2942; color:#ffffff;'>
        <th style='padding:8px; border:1px solid #cccccc;'>Analytical Stage</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Method / Specification</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Justification & Alignment with Objective</th>
      </tr>
    </thead>
    <tbody>
      <tr style='background-color:#f2f6fa; color:#111111;'>
        <td style='padding:8px; border:1px solid #dddddd; font-weight:bold;'>1. Train / Test Partition</td>
        <td style='padding:8px; border:1px solid #dddddd;'>80% Train ({len(train_rows)} rows) / 20% Test ({len(test_rows)} rows)</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Evaluates out-of-sample generalization on unseen patients.</td>
      </tr>
      <tr style='background-color:#ffffff; color:#111111;'>
        <td style='padding:8px; border:1px solid #dddddd; font-weight:bold;'>2. Feature Standardization</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Z-Score Scaling (&mu; = 0, &sigma; = 1) fitted on Train Set</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Ensures Ridge/Lasso penalties treat all clinical units equally without data leakage.</td>
      </tr>
      <tr style='background-color:#f2f6fa; color:#111111;'>
        <td style='padding:8px; border:1px solid #dddddd; font-weight:bold;'>3. Multicollinearity Control</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Ridge (L2), Lasso (L1), and ElasticNet (L1+L2)</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Stabilizes correlated cholesterol predictors and performs automatic feature selection.</td>
      </tr>
      <tr style='background-color:#ffffff; color:#111111;'>
        <td style='padding:8px; border:1px solid #dddddd; font-weight:bold;'>4. Non-Linear Interaction Test</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Polynomial Degree-2 Terms (BMI &times; Triglycerides, etc.)</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Tests whether compounding metabolic interactions improve predictive accuracy (R&sup2;).</td>
      </tr>
      <tr style='background-color:#f2f6fa; color:#111111;'>
        <td style='padding:8px; border:1px solid #dddddd; font-weight:bold;'>5. Evaluation Metrics</td>
        <td style='padding:8px; border:1px solid #dddddd;'>R&sup2;, Root Mean Squared Error (RMSE), and MAE</td>
        <td style='padding:8px; border:1px solid #dddddd;'>Measures both proportion of variance explained (R&sup2;) and clinical error magnitude.</td>
      </tr>
    </tbody>
  </table>
</div>
"""
display(HTML(html_obj))